# Note 3: LDL^T Factorization for Symmetric Indefinite Systems

**Goal:** Exploit symmetry ($A = A^T$) to halve work and storage, and handle **indefinite** matrices using Bunch-Kaufman pivoting.

## Why Symmetric Indefinite?

In optimization, the core linear system at each iteration is the **KKT system**:

$$\begin{bmatrix} H + \Sigma & J^T \\ J & -\delta I \end{bmatrix} \begin{bmatrix} \Delta x \\ \Delta y \end{bmatrix} = \begin{bmatrix} r_d \\ r_p \end{bmatrix}$$

This matrix is:
- **Symmetric:** by construction
- **Indefinite:** the $(1,1)$ block is positive (semi)definite, the $(2,2)$ block is negative definite

We can't use Cholesky ($A = LL^T$) because that requires positive definiteness. We need $LDL^T$.

## LDL^T Factorization

For a symmetric matrix $A$, we seek:

$$A = LDL^T$$

where:
- $L$ is unit lower triangular
- $D$ is **block diagonal** with $1 \times 1$ and $2 \times 2$ blocks

Compared to LU:
- Half the work: $\frac{1}{3}n^3$ vs $\frac{2}{3}n^3$
- Half the storage: only need $L$ (not both $L$ and $U$)
- **Inertia for free:** the signs of $D$'s eigenvalues tell us how many positive, negative, and zero eigenvalues $A$ has

### Simple Case: $1 \times 1$ Pivots

If all pivots are non-tiny, the $D$ block is purely diagonal (all $1 \times 1$):

In [ ]:
import numpy as np
np.set_printoptions(precision=6, suppress=True)

In [ ]:
def ldlt_simple(A):
    """LDL^T factorization with only 1x1 pivots (no pivoting).
    
    Works for symmetric matrices where no pivot is too small.
    """
    n = A.shape[0]
    L = np.eye(n)
    d = np.zeros(n)  # diagonal of D
    A = A.astype(float).copy()
    
    for j in range(n):
        # d[j] = A[j,j] - sum(L[j,k]^2 * d[k] for k < j)
        d[j] = A[j, j] - sum(L[j, k]**2 * d[k] for k in range(j))
        
        for i in range(j + 1, n):
            # L[i,j] = (A[i,j] - sum(L[i,k]*d[k]*L[j,k] for k < j)) / d[j]
            L[i, j] = (A[i, j] - sum(L[i, k] * d[k] * L[j, k] for k in range(j))) / d[j]
    
    return L, d

In [ ]:
# A simple symmetric indefinite matrix
A = np.array([
    [ 4.0,  2.0, -2.0],
    [ 2.0, -3.0,  1.0],
    [-2.0,  1.0,  5.0]
])

L, d = ldlt_simple(A)

print("L =")
print(L)
print(f"\nd = {d}")

# Verify: A = L D L^T
D = np.diag(d)
print(f"\n||A - L D L^T|| = {np.linalg.norm(A - L @ D @ L.T):.2e}")

# Inertia: count positive, negative, zero eigenvalues of D
n_pos = np.sum(d > 0)
n_neg = np.sum(d < 0)
n_zero = np.sum(np.abs(d) < 1e-12)
print(f"\nInertia: ({n_pos}+, {n_neg}-, {n_zero}z)")
print(f"NumPy eigenvalues: {np.sort(np.linalg.eigvalsh(A))}")

## Solving with LDL^T

Given $A = LDL^T$, solve $Ax = b$ in three steps:

1. $Ly = b$ (forward substitution)
2. $Dz = y$ (divide by diagonal entries)
3. $L^T x = z$ (back substitution)

In [ ]:
def ldlt_solve(L, d, b):
    """Solve Ax = b given A = LDL^T (1x1 pivots only)."""
    n = len(b)
    b = b.astype(float)
    
    # Forward: Ly = b
    y = np.zeros(n)
    for i in range(n):
        y[i] = b[i] - L[i, :i] @ y[:i]
    
    # Diagonal: Dz = y
    z = y / d
    
    # Backward: L^T x = z
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = z[i] - L[i+1:, i] @ x[i+1:]
    
    return x

b = np.array([1.0, -2.0, 3.0])
x = ldlt_solve(L, d, b)
print(f"Solution: {x}")
print(f"Residual: {np.linalg.norm(A @ x - b):.2e}")

## Bunch-Kaufman Pivoting

Simple LDL^T fails when a diagonal entry becomes zero or tiny during factorization. **Bunch-Kaufman pivoting** fixes this by allowing **$2 \times 2$ pivot blocks** in $D$.

### The Algorithm

At each step, choose between:
- **$1 \times 1$ pivot:** Use $a_{kk}$ if it's large enough relative to the off-diagonal entries
- **$2 \times 2$ pivot:** Group rows $k$ and some row $r$ together, putting a $2 \times 2$ block in $D$

The decision uses a threshold $\alpha = \frac{1 + \sqrt{17}}{8} \approx 0.6404$:

1. Find $\lambda = \max_{i > k} |a_{ik}|$ (largest off-diagonal in column $k$)
2. If $|a_{kk}| \geq \alpha \cdot \lambda$: use $1 \times 1$ pivot
3. Otherwise, find $\sigma = \max_{j \neq r} |a_{rj}|$ where $r$ is the row of $\lambda$
4. If $|a_{kk}| \cdot \sigma \geq \alpha \cdot \lambda^2$: use $1 \times 1$ pivot
5. If $|a_{rr}| \geq \alpha \cdot \sigma$: swap $r$ to position $k$, use $1 \times 1$ pivot
6. Otherwise: use $2 \times 2$ pivot on rows/cols $\{k, r\}$

In [ ]:
def ldlt_bunch_kaufman(A):
    """LDL^T with Bunch-Kaufman pivoting.
    
    Returns L, D (block diagonal), perm.
    D is stored as a list of blocks: each is either a scalar or a 2x2 array.
    """
    n = A.shape[0]
    A = A.astype(float).copy()
    L = np.eye(n)
    perm = list(range(n))
    D_blocks = []  # list of (size, block_data)
    alpha = (1.0 + np.sqrt(17.0)) / 8.0
    
    k = 0
    while k < n:
        if k == n - 1:
            # Last entry: must be 1x1
            D_blocks.append((1, A[k, k]))
            k += 1
            continue
        
        # Find largest off-diagonal in column k (below diagonal)
        col_below = np.abs(A[k+1:, k])
        if len(col_below) == 0:
            D_blocks.append((1, A[k, k]))
            k += 1
            continue
        
        r_rel = np.argmax(col_below)
        lam = col_below[r_rel]
        r = k + 1 + r_rel  # absolute index
        
        use_1x1 = False
        use_2x2 = False
        
        if lam < 1e-15:
            # Column is zero below diagonal — 1x1 pivot
            use_1x1 = True
        elif abs(A[k, k]) >= alpha * lam:
            # Test 1: diagonal is large enough
            use_1x1 = True
        else:
            # Find sigma: largest off-diagonal in row r
            sigma = 0.0
            for j in range(n):
                if j != r:
                    sigma = max(sigma, abs(A[r, j]))
            
            if abs(A[k, k]) * sigma >= alpha * lam * lam:
                use_1x1 = True
            elif abs(A[r, r]) >= alpha * sigma:
                # Swap r to position k, then 1x1
                if r != k:
                    A[[k, r]] = A[[r, k]]
                    A[:, [k, r]] = A[:, [r, k]]
                    L[[k, r], :k] = L[[r, k], :k]
                    perm[k], perm[r] = perm[r], perm[k]
                use_1x1 = True
            else:
                # Swap r to position k+1, then 2x2
                if r != k + 1:
                    A[[k+1, r]] = A[[r, k+1]]
                    A[:, [k+1, r]] = A[:, [r, k+1]]
                    L[[k+1, r], :k] = L[[r, k+1], :k]
                    perm[k+1], perm[r] = perm[r], perm[k+1]
                use_2x2 = True
        
        if use_1x1:
            pivot = A[k, k]
            D_blocks.append((1, pivot))
            
            if abs(pivot) > 1e-15:
                for i in range(k + 1, n):
                    L[i, k] = A[i, k] / pivot
                
                # Update trailing submatrix
                for i in range(k + 1, n):
                    for j in range(k + 1, i + 1):
                        A[i, j] -= L[i, k] * pivot * L[j, k]
                        A[j, i] = A[i, j]
            k += 1
        
        elif use_2x2:
            # 2x2 pivot block
            E = np.array([[A[k, k], A[k+1, k]],
                          [A[k+1, k], A[k+1, k+1]]])
            D_blocks.append((2, E.copy()))
            
            # Compute E^{-1}
            det = E[0, 0] * E[1, 1] - E[0, 1] * E[1, 0]
            E_inv = np.array([[E[1, 1], -E[0, 1]],
                              [-E[1, 0], E[0, 0]]]) / det
            
            # L entries for rows below k+1
            for i in range(k + 2, n):
                v = np.array([A[i, k], A[i, k+1]])
                l_row = E_inv @ v
                L[i, k] = l_row[0]
                L[i, k+1] = l_row[1]
            
            # Update trailing submatrix
            for i in range(k + 2, n):
                for j in range(k + 2, i + 1):
                    li = np.array([L[i, k], L[i, k+1]])
                    lj = np.array([L[j, k], L[j, k+1]])
                    A[i, j] -= li @ E @ lj
                    A[j, i] = A[i, j]
            k += 2
    
    return L, D_blocks, perm

In [ ]:
def ldlt_bk_solve(L, D_blocks, perm, b):
    """Solve Ax = b given Bunch-Kaufman LDL^T factorization."""
    n = len(b)
    # Apply permutation
    pb = b[perm].astype(float)
    
    # Forward: Ly = Pb
    y = np.zeros(n)
    for i in range(n):
        y[i] = pb[i] - L[i, :i] @ y[:i]
    
    # Diagonal solve: Dz = y (block-wise)
    z = np.zeros(n)
    idx = 0
    for size, block in D_blocks:
        if size == 1:
            z[idx] = y[idx] / block
            idx += 1
        else:  # 2x2
            rhs = y[idx:idx+2]
            z[idx:idx+2] = np.linalg.solve(block, rhs)
            idx += 2
    
    # Backward: L^T x_perm = z
    x_perm = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x_perm[i] = z[i] - L[i+1:, i] @ x_perm[i+1:]
    
    # Undo permutation
    x = np.zeros(n)
    for i, p in enumerate(perm):
        x[p] = x_perm[i]
    
    return x

In [ ]:
# Test on a matrix where simple LDL^T would need a zero pivot
A_tricky = np.array([
    [0.0,  1.0,  2.0],
    [1.0,  0.0,  3.0],
    [2.0,  3.0,  1.0]
])

L, D_blocks, perm = ldlt_bunch_kaufman(A_tricky)

print("D blocks:")
for i, (size, block) in enumerate(D_blocks):
    if size == 1:
        print(f"  1x1: {block:.6f}")
    else:
        print(f"  2x2:\n{block}")

b = np.array([1.0, 2.0, 3.0])
x = ldlt_bk_solve(L, D_blocks, perm, b)
print(f"\nSolution: {x}")
print(f"Residual: {np.linalg.norm(A_tricky @ x - b):.2e}")
print(f"NumPy:    {np.linalg.solve(A_tricky, b)}")

## Inertia: The Free Bonus

The **inertia** of a symmetric matrix is the triple $(n_+, n_-, n_0)$: the number of positive, negative, and zero eigenvalues.

From $A = LDL^T$, inertia comes directly from $D$:
- Each positive $d_i$ contributes to $n_+$
- Each negative $d_i$ contributes to $n_-$
- Each $2 \times 2$ block contributes one to $n_+$ and one to $n_-$ (by construction, BK only creates $2 \times 2$ blocks with one positive and one negative eigenvalue)

This is critical in optimization: the KKT system should have inertia $(n, m, 0)$ — if it doesn't, the solver adds regularization.

In [ ]:
def compute_inertia(D_blocks):
    """Compute inertia (n+, n-, n0) from Bunch-Kaufman D blocks."""
    n_pos, n_neg, n_zero = 0, 0, 0
    for size, block in D_blocks:
        if size == 1:
            if abs(block) < 1e-12:
                n_zero += 1
            elif block > 0:
                n_pos += 1
            else:
                n_neg += 1
        else:  # 2x2 block always has one + and one - eigenvalue
            eigs = np.linalg.eigvalsh(block)
            for e in eigs:
                if abs(e) < 1e-12:
                    n_zero += 1
                elif e > 0:
                    n_pos += 1
                else:
                    n_neg += 1
    return n_pos, n_neg, n_zero


# Example: a KKT-like matrix
# [H  J^T]   with H positive definite, J full rank
# [J  0  ]

H = np.array([[4.0, 1.0],
              [1.0, 3.0]])  # 2x2, positive definite
J = np.array([[1.0, 2.0]])  # 1x2 constraint Jacobian

KKT = np.block([
    [H, J.T],
    [J, np.zeros((1, 1))]
])

print("KKT matrix:")
print(KKT)

L, D_blocks, perm = ldlt_bunch_kaufman(KKT)
inertia = compute_inertia(D_blocks)
print(f"\nInertia: ({inertia[0]}+, {inertia[1]}-, {inertia[2]}z)")
print(f"Expected for n=2 vars, m=1 constraint: (2+, 1-, 0z)")
print(f"\nActual eigenvalues: {np.sort(np.linalg.eigvalsh(KKT))}")

## Packed Storage

Since $A$ is symmetric, we only need to store the lower (or upper) triangle. This halves memory from $n^2$ to $\frac{n(n+1)}{2}$.

**Column-major packed format** stores the lower triangle column by column:

```
  [a00          ]     Packed: [a00, a10, a20, a11, a21, a22]
  [a10  a11     ]     Index:  j*n - j*(j+1)/2 + i  (for i >= j)
  [a20  a21  a22]
```

This is exactly what ripopt's `SymmetricMatrix` uses in `src/linear_solver/mod.rs`.

In [ ]:
class PackedSymmetric:
    """Symmetric matrix in packed column-major lower-triangle storage."""
    
    def __init__(self, n):
        self.n = n
        self.data = np.zeros(n * (n + 1) // 2)
    
    def _idx(self, i, j):
        if i < j:
            i, j = j, i  # ensure i >= j (lower triangle)
        return j * self.n - j * (j + 1) // 2 + i
    
    def get(self, i, j):
        return self.data[self._idx(i, j)]
    
    def set(self, i, j, val):
        self.data[self._idx(i, j)] = val
    
    def to_dense(self):
        A = np.zeros((self.n, self.n))
        for i in range(self.n):
            for j in range(i + 1):
                A[i, j] = A[j, i] = self.get(i, j)
        return A


# Demo
M = PackedSymmetric(3)
M.set(0, 0, 4.0)
M.set(1, 0, 2.0)
M.set(1, 1, -3.0)
M.set(2, 0, -2.0)
M.set(2, 1, 1.0)
M.set(2, 2, 5.0)

print(f"Packed data ({len(M.data)} entries vs {3*3}=9 for full):")
print(M.data)
print(f"\nRecovered matrix:\n{M.to_dense()}")

## What We've Learned

1. **LDL^T factorization** exploits symmetry to halve work ($\frac{1}{3}n^3$) and storage ($\frac{n(n+1)}{2}$)
2. **Bunch-Kaufman pivoting** handles indefinite matrices by allowing $2 \times 2$ blocks in $D$
3. **Inertia** (the signature of $D$) comes for free and is essential for optimization
4. **Packed storage** eliminates redundancy in symmetric matrices

## What's Next

Everything so far is $O(n^3)$ and stores the entire matrix densely. Real optimization problems have **sparse** matrices — most entries are zero. In Note 4, we introduce sparse matrix formats (COO, CSC) that store only the nonzeros, and see how sparsity reduces both storage and computation dramatically.

---

*This is Note 3 in a series building from basic Gaussian elimination to the multifrontal sparse solver used in [ripopt](https://github.com/jkitchin/ripopt).*